In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
 !pip install transformers accelerate

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from tqdm import tqdm

torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

In [5]:
train_df = pd.read_csv("/kaggle/input/datasets/palakjaiswal24/datasetfinal/train_final.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/palakjaiswal24/datasetfinal/val_final.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/palakjaiswal24/datasetfinal/test_final.csv")

print(len(train_df), len(val_df), len(test_df))

81248 10156 10157


In [6]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [29]:
MAX_LENGTH = 256
STRIDE = 128
MAX_CHUNKS = 10   # Balanced for speed + performance

In [33]:
class HierarchicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            truncation=True,
            padding="max_length",
            return_overflowing_tokens=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]

        # Limit chunks
        if input_ids.size(0) > MAX_CHUNKS:
            input_ids = input_ids[:MAX_CHUNKS]
            attention_mask = attention_mask[:MAX_CHUNKS]

        # Pad chunks if needed
        pad_chunks = MAX_CHUNKS - input_ids.size(0)

        if pad_chunks > 0:
            pad_input = torch.zeros((pad_chunks, MAX_LENGTH), dtype=torch.long)
            pad_mask = torch.zeros((pad_chunks, MAX_LENGTH), dtype=torch.long)

            input_ids = torch.cat([input_ids, pad_input], dim=0)
            attention_mask = torch.cat([attention_mask, pad_mask], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.long)
        }

In [34]:
train_dataset = HierarchicalDataset(train_df, tokenizer)
val_dataset   = HierarchicalDataset(val_df, tokenizer)
test_dataset  = HierarchicalDataset(test_df, tokenizer)

BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)

In [37]:
class HierarchicalXLMRBase(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained("xlm-roberta-base")
        self.encoder.gradient_checkpointing_enable()

        hidden_size = 768

        self.doc_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_size,
                nhead=8,
                dim_feedforward=2048,
                dropout=0.1,
                batch_first=True
            ),
            num_layers=2
        )

        self.attention = nn.Linear(hidden_size, 1)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, input_ids, attention_mask):
        batch_size, num_chunks, seq_len = input_ids.size()

        input_ids = input_ids.view(-1, seq_len)
        attention_mask = attention_mask.view(-1, seq_len)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)

        doc_outputs = self.doc_transformer(cls_embeddings)

        attn_weights = torch.softmax(self.attention(doc_outputs), dim=1)
        doc_rep = torch.sum(attn_weights * doc_outputs, dim=1)

        logits = self.classifier(doc_rep)

        return logits

In [38]:
model = HierarchicalXLMRBase().cuda()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
EPOCHS_PHASE1 = 1
EPOCHS_PHASE2 = 2

GRAD_ACCUM = 8

LR_PHASE1 = 1e-5
LR_PHASE2 = 5e-6
WEIGHT_DECAY = 0.01

In [44]:
total_steps_phase1 = len(train_loader) * EPOCHS_PHASE1 // GRAD_ACCUM

optimizer = AdamW(model.parameters(), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps_phase1),
    num_training_steps=total_steps_phase1
)

scaler = GradScaler("cuda")

In [45]:
for name, param in model.encoder.named_parameters():
    if "encoder.layer." in name:
        layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
        if layer_num < 4:
            param.requires_grad = False

In [48]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader)):

        input_ids = batch["input_ids"].cuda(non_blocking=True)
        attention_mask = batch["attention_mask"].cuda(non_blocking=True)
        labels = batch["label"].cuda(non_blocking=True)

        with autocast("cuda"):
            outputs = model(input_ids, attention_mask)
            loss = F.cross_entropy(outputs, labels)

        scaler.scale(loss / GRAD_ACCUM).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].cuda(non_blocking=True)
            attention_mask = batch["attention_mask"].cuda(non_blocking=True)
            labels = batch["label"].cuda(non_blocking=True)

            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [49]:
best_val_acc = 0

for epoch in range(EPOCHS_PHASE1):
    print(f"\nPhase 1 - Epoch {epoch+1}")

    train_loss = train_epoch(model, train_loader)
    val_acc = eval_epoch(model, val_loader)

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")
        print("Best model saved.")


Phase 1 - Epoch 1


100%|██████████| 40624/40624 [3:07:46<00:00,  3.61it/s]  


Train Loss: 0.46808394730419756
Validation Accuracy: 0.8934619929105947
Best model saved.


In [53]:
for param in model.encoder.parameters():
    param.requires_grad = True

total_steps_phase2 = len(train_loader) * EPOCHS_PHASE2 // GRAD_ACCUM

optimizer = AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps_phase2),
    num_training_steps=total_steps_phase2
)

In [54]:
for epoch in range(EPOCHS_PHASE2):
    print(f"\nPhase 2 - Epoch {epoch+1}")

    train_loss = train_epoch(model, train_loader)
    val_acc = eval_epoch(model, val_loader)

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")
        print("Best model saved.")


Phase 2 - Epoch 1


100%|██████████| 40624/40624 [3:19:11<00:00,  3.40it/s]  


Train Loss: 0.2513320890584298
Validation Accuracy: 0.9059669161087042
Best model saved.

Phase 2 - Epoch 2


100%|██████████| 40624/40624 [3:18:52<00:00,  3.40it/s]  


Train Loss: 0.21683059072590757
Validation Accuracy: 0.907345411579362
Best model saved.
